# 1. Data Cleaning and Preprocessing

In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('C:\\Tech notes, prac, etc\\Data Analytics and Visulization Lab\\Major Project\\data\\processed\\weather\\city_wise_10yr_clean.csv')

In [4]:
df.head()

,city,country,continent,latitude,longitude,temperature_2m_max_2015-01-01,temperature_2m_max_2015-01-02,temperature_2m_max_2015-01-03,temperature_2m_max_2015-01-04,temperature_2m_max_2015-01-05,...,temperature_anomaly_2024-12-22,temperature_anomaly_2024-12-23,temperature_anomaly_2024-12-24,temperature_anomaly_2024-12-25,temperature_anomaly_2024-12-26,temperature_anomaly_2024-12-27,temperature_anomaly_2024-12-28,temperature_anomaly_2024-12-29,temperature_anomaly_2024-12-30,temperature_anomaly_2024-12-31
0,Berlin,Germany,Europe,52.5200,13.4050,4.4,7.5,5.1,4.2,3.7,...,-5.484561,-7.684561,-7.484561,-8.584561,-5.584561,-7.284561,-12.084561,-11.884561,-8.384561,-9.184561
1,Buenos Aires,Argentina,South America,-34.6037,-58.3816,22.8,23.6,27.1,23.6,30.3,...,4.519956,7.419956,4.419956,0.819956,2.719956,4.719956,4.319956,4.619956,6.019956,7.519956
2,Cairo,Egypt,Africa,30.0444,31.2357,15.5,17.6,16.9,17.6,17.8,...,-6.797947,-7.397947,-8.597947,-7.897947,-9.997947,-9.897947,-9.397947,-8.797947,-9.297947,-8.797947
3,Cape Town,South Africa,Africa,-33.9249,18.4241,25.0,24.1,24.3,24.1,22.6,...,4.096496,2.996496,4.296496,7.996496,6.296496,3.596496,5.796496,2.596496,1.296496,1.896496
4,Delhi,India,Asia,28.6139,77.2090,21.3,17.2,17.9,19.1,20.7,...,-8.233206,-10.033206,-9.833206,-9.433206,-8.433206,-9.433206,-9.633206,-10.833206,-11.833206,-13.233206


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Columns: 14617 entries, city to temperature_anomaly_2024-12-31
dtypes: float64(14614), object(3)
memory usage: 1.7+ MB


In [6]:
df.shape

(15, 14617)

In [7]:
df.columns = [col.replace("temperature_2m_", "temp_") for col in df.columns]

In [ ]:
meta_cols = ["city", "country", "continent", "latitude", "longitude"]

mean_cols = [c for c in df.columns if c.startswith("temp_mean_")]
max_cols = [c for c in df.columns if c.startswith("temp_max_")]
min_cols = [c for c in df.columns if c.startswith("temp_min_")]

mean_cols = sorted(mean_cols)
max_cols = sorted(max_cols)
min_cols = sorted(min_cols)

print("Mean columns:", len(mean_cols))
print("Max columns :", len(max_cols))
print("Min columns :", len(min_cols))

if len(mean_cols) == 0 or len(max_cols) == 0 or len(min_cols) == 0:
    print("Temperature columns not found. Check column names once.")


In [ ]:
print("Metadata columns:", meta_cols)
print("First mean column:", mean_cols[0])
print("First max column :", max_cols[0])
print("First min column :", min_cols[0])


In [ ]:
print("Date range:", mean_cols[0][-10:], "to", mean_cols[-1][-10:])


In [ ]:
df_mean = df[meta_cols + mean_cols].melt(
    id_vars=meta_cols,
    var_name="date",
    value_name="temp_mean"
)

df_mean["date"] = df_mean["date"].str.replace("temp_mean_", "", regex=False)
df_mean["date"] = pd.to_datetime(df_mean["date"])

df_mean.head()


In [ ]:
df_max = df[["city"] + max_cols].melt(
    id_vars=["city"],
    var_name="date",
    value_name="temp_max"
)

df_max["date"] = df_max["date"].str.replace("temp_max_", "", regex=False)
df_max["date"] = pd.to_datetime(df_max["date"])

df_max.head()


In [ ]:
df_min = df[["city"] + min_cols].melt(
    id_vars=["city"],
    var_name="date",
    value_name="temp_min"
)

df_min["date"] = df_min["date"].str.replace("temp_min_", "", regex=False)
df_min["date"] = pd.to_datetime(df_min["date"])

df_min.head()


In [ ]:
df = pd.merge(df_mean, df_max, on=["city", "date"], how="left")
df = pd.merge(df, df_min, on=["city", "date"], how="left")

df = df.sort_values(["city", "date"]).reset_index(drop=True)
df.head()


In [ ]:
print("Long format shape:", df.shape)
print("Number of cities:", df["city"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())


In [16]:
df

,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min
0,Berlin,Germany,Europe,52.5200,13.4050,2.8,2015-01-01,4.4,0.9
1,Buenos Aires,Argentina,South America,-34.6037,-58.3816,19.8,2015-01-01,22.8,15.9
2,Cairo,Egypt,Africa,30.0444,31.2357,12.3,2015-01-01,15.5,7.9
3,Cape Town,South Africa,Africa,-33.9249,18.4241,21.6,2015-01-01,25.0,18.7
4,Delhi,India,Asia,28.6139,77.2090,16.3,2015-01-01,21.3,12.0
...,...,...,...,...,...,...,...,...,...
54790,Sao Paulo,Brazil,South America,-23.5505,-46.6333,21.9,2024-12-31,28.0,17.8
54791,Singapore,Singapore,Asia,1.3521,103.8198,25.7,2024-12-31,31.5,23.0
54792,Sydney,Australia,Australia,-33.8688,151.2093,23.2,2024-12-31,26.6,18.9
54793,Tokyo,Japan,Asia,35.6762,139.6503,6.7,2024-12-31,12.8,0.9


## Basic datatime features

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["day_of_year"] = df["date"].dt.dayofyear


In [18]:
df.head()

,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min,year,month,day_of_year
0,Berlin,Germany,Europe,52.5200,13.4050,2.8,2015-01-01,4.4,0.9,2015,1,1
1,Buenos Aires,Argentina,South America,-34.6037,-58.3816,19.8,2015-01-01,22.8,15.9,2015,1,1
2,Cairo,Egypt,Africa,30.0444,31.2357,12.3,2015-01-01,15.5,7.9,2015,1,1
3,Cape Town,South Africa,Africa,-33.9249,18.4241,21.6,2015-01-01,25.0,18.7,2015,1,1
4,Delhi,India,Asia,28.6139,77.2090,16.3,2015-01-01,21.3,12.0,2015,1,1


In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["season"] = df["month"].apply(get_season)
df.head()


In [20]:
df.head()

,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min,year,month,day_of_year,season
0,Berlin,Germany,Europe,52.5200,13.4050,2.8,2015-01-01,4.4,0.9,2015,1,1,Winter
1,Buenos Aires,Argentina,South America,-34.6037,-58.3816,19.8,2015-01-01,22.8,15.9,2015,1,1,Winter
2,Cairo,Egypt,Africa,30.0444,31.2357,12.3,2015-01-01,15.5,7.9,2015,1,1,Winter
3,Cape Town,South Africa,Africa,-33.9249,18.4241,21.6,2015-01-01,25.0,18.7,2015,1,1,Winter
4,Delhi,India,Asia,28.6139,77.2090,16.3,2015-01-01,21.3,12.0,2015,1,1,Winter


In [24]:
df[["temp_mean", "temp_max", "temp_min"]].isnull().sum()

temp_mean    0
temp_max     0
temp_min     0
dtype: int64

## No missing values

In [25]:
for col in ["temp_mean", "temp_max", "temp_min"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [26]:
df["hemisphere"] = np.where(df["latitude"] >= 0, "Northern", "Southern")
df["temp_range_day_wise"] = df["temp_max"] - df["temp_min"]

df.head()


,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min,year,month,day_of_year,season,hemisphere,temp_range_day_wise
0,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-01,4.4,0.9,2015,1,1,Winter,Northern,3.5
1,Berlin,Germany,Europe,52.52,13.405,4.6,2015-01-02,7.5,1.6,2015,1,2,Winter,Northern,5.9
2,Berlin,Germany,Europe,52.52,13.405,3.9,2015-01-03,5.1,2.9,2015,1,3,Winter,Northern,2.2
3,Berlin,Germany,Europe,52.52,13.405,3.0,2015-01-04,4.2,1.6,2015,1,4,Winter,Northern,2.6
4,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-05,3.7,1.1,2015,1,5,Winter,Northern,2.6


In [27]:
start_year = df["year"].min()
end_year = start_year + 4

In [28]:
baseline_df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]
city_baseline = baseline_df.groupby("city")["temp_mean"].mean().reset_index()
city_baseline = city_baseline.rename(columns={"temp_mean": "baseline_city_temp"})

In [30]:
df = pd.merge(df, city_baseline, on="city", how="left")
df["temp_anomaly"] = df["temp_mean"] - df["baseline_city_temp"]


In [33]:
print("Baseline period:", start_year, "to", end_year)
df[["city", "date", "temp_mean", "baseline_city_temp", "temp_anomaly"]].head()


Baseline period: 2015 to 2019


,city,date,temp_mean,baseline_city_temp,temp_anomaly
0,Berlin,2015-01-01,2.8,10.780723,-7.980723
1,Berlin,2015-01-02,4.6,10.780723,-6.180723
2,Berlin,2015-01-03,3.9,10.780723,-6.880723
3,Berlin,2015-01-04,3.0,10.780723,-7.780723
4,Berlin,2015-01-05,2.8,10.780723,-7.980723


In [34]:
df

,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min,year,month,day_of_year,season,hemisphere,temp_range_day_wise,baseline_city_temp,temp_anomaly
0,Berlin,Germany,Europe,52.5200,13.4050,2.8,2015-01-01,4.4,0.9,2015,1,1,Winter,Northern,3.5,10.780723,-7.980723
1,Berlin,Germany,Europe,52.5200,13.4050,4.6,2015-01-02,7.5,1.6,2015,1,2,Winter,Northern,5.9,10.780723,-6.180723
2,Berlin,Germany,Europe,52.5200,13.4050,3.9,2015-01-03,5.1,2.9,2015,1,3,Winter,Northern,2.2,10.780723,-6.880723
3,Berlin,Germany,Europe,52.5200,13.4050,3.0,2015-01-04,4.2,1.6,2015,1,4,Winter,Northern,2.6,10.780723,-7.780723
4,Berlin,Germany,Europe,52.5200,13.4050,2.8,2015-01-05,3.7,1.1,2015,1,5,Winter,Northern,2.6,10.780723,-7.980723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54790,Toronto,Canada,North America,43.6532,-79.3832,1.0,2024-12-27,2.3,-0.1,2024,12,362,Winter,Northern,2.4,8.934173,-7.934173
54791,Toronto,Canada,North America,43.6532,-79.3832,5.6,2024-12-28,9.4,2.4,2024,12,363,Winter,Northern,7.0,8.934173,-3.334173
54792,Toronto,Canada,North America,43.6532,-79.3832,5.1,2024-12-29,9.9,2.2,2024,12,364,Winter,Northern,7.7,8.934173,-3.834173
54793,Toronto,Canada,North America,43.6532,-79.3832,4.9,2024-12-30,9.8,2.3,2024,12,365,Winter,Northern,7.5,8.934173,-4.034173


In [35]:
df["temp_7day_avg"] = df.groupby("city")["temp_mean"].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)

df["temp_30day_avg"] = df.groupby("city")["temp_mean"].transform(
    lambda x: x.rolling(30, min_periods=1).mean()
)

df.head()


,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min,year,month,day_of_year,season,hemisphere,temp_range_day_wise,baseline_city_temp,temp_anomaly,temp_7day_avg,temp_30day_avg
0,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-01,4.4,0.9,2015,1,1,Winter,Northern,3.5,10.780723,-7.980723,2.800000,2.800000
1,Berlin,Germany,Europe,52.52,13.405,4.6,2015-01-02,7.5,1.6,2015,1,2,Winter,Northern,5.9,10.780723,-6.180723,3.700000,3.700000
2,Berlin,Germany,Europe,52.52,13.405,3.9,2015-01-03,5.1,2.9,2015,1,3,Winter,Northern,2.2,10.780723,-6.880723,3.766667,3.766667
3,Berlin,Germany,Europe,52.52,13.405,3.0,2015-01-04,4.2,1.6,2015,1,4,Winter,Northern,2.6,10.780723,-7.780723,3.575000,3.575000
4,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-05,3.7,1.1,2015,1,5,Winter,Northern,2.6,10.780723,-7.980723,3.420000,3.420000


## Some more features

In [36]:
df["city_yearly_mean"] = df.groupby(["city", "year"])["temp_mean"].transform("mean")
df["continent_yearly_mean"] = df.groupby(["continent", "year"])["temp_mean"].transform("mean")
df["global_daily_mean"] = df.groupby("date")["temp_mean"].transform("mean")

df.head()


,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min,year,...,season,hemisphere,temp_range_day_wise,baseline_city_temp,temp_anomaly,temp_7day_avg,temp_30day_avg,city_yearly_mean,continent_yearly_mean,global_daily_mean
0,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-01,4.4,0.9,2015,...,Winter,Northern,3.5,10.780723,-7.980723,2.800000,2.800000,10.834521,11.363288,11.780000
1,Berlin,Germany,Europe,52.52,13.405,4.6,2015-01-02,7.5,1.6,2015,...,Winter,Northern,5.9,10.780723,-6.180723,3.700000,3.700000,10.834521,11.363288,12.333333
2,Berlin,Germany,Europe,52.52,13.405,3.9,2015-01-03,5.1,2.9,2015,...,Winter,Northern,2.2,10.780723,-6.880723,3.766667,3.766667,10.834521,11.363288,12.333333
3,Berlin,Germany,Europe,52.52,13.405,3.0,2015-01-04,4.2,1.6,2015,...,Winter,Northern,2.6,10.780723,-7.780723,3.575000,3.575000,10.834521,11.363288,12.926667
4,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-05,3.7,1.1,2015,...,Winter,Northern,2.6,10.780723,-7.980723,3.420000,3.420000,10.834521,11.363288,12.386667


In [40]:
annual_city = df.groupby(["city", "country", "continent", "hemisphere", "year"]).agg({
    "temp_mean": "mean",
    "temp_max": ["mean", "max"],
    "temp_min": ["mean", "min"],
    "temp_range_day_wise": "mean",
    "temp_anomaly": "mean",
    "latitude": "first",
    "longitude": "first"
}).reset_index()

annual_city.columns = [
    "city", "country", "continent", "hemisphere", "year",
    "temp_annual_mean", "temp_annual_avg_max", "temp_annual_record_max",
    "temp_annual_avg_min", "temp_annual_record_min",
    "temp_range_mean", "anomaly_annual_mean", "latitude", "longitude"
]

annual_city.head()


,city,country,continent,hemisphere,year,temp_annual_mean,temp_annual_avg_max,temp_annual_record_max,temp_annual_avg_min,temp_annual_record_min,temp_range_mean,anomaly_annual_mean,latitude,longitude
0,Berlin,Germany,Europe,Northern,2015,10.834521,14.581370,36.1,7.053425,-8.2,7.527945,0.053798,52.52,13.405
1,Berlin,Germany,Europe,Northern,2016,10.464481,14.072131,34.7,6.785792,-12.8,7.286339,-0.316242,52.52,13.405
2,Berlin,Germany,Europe,Northern,2017,9.966575,13.504110,30.0,6.498630,-10.6,7.005479,-0.814148,52.52,13.405
3,Berlin,Germany,Europe,Northern,2018,11.186849,15.338630,36.6,7.104932,-12.4,8.233699,0.406126,52.52,13.405
4,Berlin,Germany,Europe,Northern,2019,11.452055,15.510959,37.3,7.430685,-7.2,8.080274,0.671332,52.52,13.405


In [43]:
global_annual = df.groupby("year").agg({
    "temp_mean": "mean",
    "temp_max": ["mean", "max"],
    "temp_min": ["mean", "min"],
    "temp_anomaly": "mean"
}).reset_index()

global_annual.columns = [
    "year", "global_temp_mean", "global_temp_avg_max", "global_temp_record_max",
    "global_temp_avg_min", "global_temp_record_min", "global_anomaly_mean"
]

global_annual.head()


,year,global_temp_mean,global_temp_avg_max,global_temp_record_max,global_temp_avg_min,global_temp_record_min,global_anomaly_mean
0,2015,16.961808,21.325260,45.5,12.961735,-24.3,0.003758
1,2016,17.074900,21.453971,45.1,13.085665,-23.2,0.116849
2,2017,16.909845,21.539689,44.0,12.888877,-24.8,-0.048206
3,2018,16.975945,21.616822,45.1,12.909516,-24.5,0.017895
4,2019,16.867434,21.551671,44.3,12.749260,-21.9,-0.090617


In [45]:
df.columns

Index(['city', 'country', 'continent', 'latitude', 'longitude', 'temp_mean',
       'date', 'temp_max', 'temp_min', 'year', 'month', 'day_of_year',
       'season', 'hemisphere', 'temp_range_day_wise', 'baseline_city_temp',
       'temp_anomaly', 'temp_7day_avg', 'temp_30day_avg', 'city_yearly_mean',
       'continent_yearly_mean', 'global_daily_mean'],
      dtype='object')

In [46]:
df.head()


,city,country,continent,latitude,longitude,temp_mean,date,temp_max,temp_min,year,...,season,hemisphere,temp_range_day_wise,baseline_city_temp,temp_anomaly,temp_7day_avg,temp_30day_avg,city_yearly_mean,continent_yearly_mean,global_daily_mean
0,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-01,4.4,0.9,2015,...,Winter,Northern,3.5,10.780723,-7.980723,2.800000,2.800000,10.834521,11.363288,11.780000
1,Berlin,Germany,Europe,52.52,13.405,4.6,2015-01-02,7.5,1.6,2015,...,Winter,Northern,5.9,10.780723,-6.180723,3.700000,3.700000,10.834521,11.363288,12.333333
2,Berlin,Germany,Europe,52.52,13.405,3.9,2015-01-03,5.1,2.9,2015,...,Winter,Northern,2.2,10.780723,-6.880723,3.766667,3.766667,10.834521,11.363288,12.333333
3,Berlin,Germany,Europe,52.52,13.405,3.0,2015-01-04,4.2,1.6,2015,...,Winter,Northern,2.6,10.780723,-7.780723,3.575000,3.575000,10.834521,11.363288,12.926667
4,Berlin,Germany,Europe,52.52,13.405,2.8,2015-01-05,3.7,1.1,2015,...,Winter,Northern,2.6,10.780723,-7.980723,3.420000,3.420000,10.834521,11.363288,12.386667


In [47]:
df.to_csv("data/processed/weather/" + "master_long.csv", index=False)
annual_city.to_csv("data/processed/weather/" + "annual_city.csv", index=False)
global_annual.to_csv("data/processed/weather/" + "global_annual.csv", index=False)


## Data Cleaning done